In [1]:
import shutil
import ctypes
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

In [2]:
ctypes.windll.kernel32.SetThreadExecutionState(0x80000002)

-2147483648

In [3]:
yolo_data="./annotations/YOLO_UAV"
img="./annotations/YOLO_UAV/images"

In [4]:
for i in ["train","val","test"]:
    os.makedirs(os.path.join(img,i),exist_ok=True)
for i in ["train","val","test"]:
    txt_fi=os.path.join(yolo_data,i+".txt")
    with open (txt_fi,'r') as f:
        lines=f.readlines()
    for l in lines:
        img_name=l.strip().split('/')[-1]
        scr_path=os.path.join(img,img_name)
        dst_path=os.path.join(img,i,img_name)
        if os.path.exists(dst_path):
            continue

        if os.path.exists(scr_path):
            shutil.move(scr_path,dst_path)

        elif os.path.exists(scr_path+'.jpg'):
            shutil.move(scr_path+'.jpg',dst_path)

print("done")

done


In [5]:
labels="./annotations/YOLO_UAV/labels"

for i in["train","val","test"]:
    os.makedirs(os.path.join(labels,i),exist_ok=True)

for i in ["train","val","test"]:
    txt_fi=os.path.join(yolo_data,i+".txt")
    with open (txt_fi,'r') as f:
        lines=f.readlines()
    for l in lines:
        img_name=l.strip().split('/')[-1]
        label_name=os.path.splitext(img_name)[0]+".txt"

        scr_path=os.path.join(labels,label_name)
        dst_path=os.path.join(labels,i,label_name)
        if os.path.exists(dst_path):
            continue

        if os.path.exists(scr_path):
            shutil.move(scr_path,dst_path)

print("done")

done


In [6]:
model=YOLO("yolov8s.pt")

In [7]:
data_yml=data='D://project_deeplearning/annotations/YOLO_UAV/data.yaml'

In [8]:
training=model.train(data=data_yml,epochs=100,imgsz=416,batch=2,workers=0,device="0",cache=True,patience=15,name='fire_detection',plots=False)

New https://pypi.org/project/ultralytics/8.4.90 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.70  Python-3.10.6 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D://project_deeplearning/annotations/YOLO_UAV/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, mome

In [10]:
model_path='D:/project_deeplearning/runs/detect/fire_detection-8/weights/best.pt'
model=YOLO(model_path)

In [19]:
img_medium =cv2.imread('D://project_deeplearning/annotations/YOLO_UAV/images/test/1.jpg')
h,w=img_medium.shape[:2]
re=model(img_medium,conf=0.5)
fire_areas = []
for i in re:
    boxes = i.boxes.xyxy.cpu().numpy()
    cls = i.boxes.cls.cpu().numpy()
    for boxx, c in zip(boxes, cls):
        if int(c) == 0:
            x1, y1, x2, y2 = map(int, boxx)
            fire_areas.append((x2 - x1) * (y2 - y1))

    if fire_areas:
        total_pct=(sum(fire_areas) / (h * w)) * 100
        if total_pct>5:
            severity="very dangerous and intense!!!!!!!"
            color=(0,0,255)
            msg="DANGER:large fire! evacuate immediately!((dispatch AIRTANKER))"
        elif total_pct>1:
            severity="medium"
            color=(0,165,225)
            msg="WARNING:medium fire!call fire department!((AIRTANKER standby))"

        else:
            severity="small fire"
            color=(0,225,0)
            msg="SMALL:notify the forester and responsible persons"

        print(f"severity: {severity}")
        print(f"percentage: {total_pct:.2f}")
        print(f"message: {msg}\n")


0: 256x416 6 fires, 1 smoke, 50.6ms
Speed: 4.4ms preprocess, 50.6ms inference, 1.4ms postprocess per image at shape (1, 3, 256, 416)
severity: very dangerous and intense!!!!!!!
percentage: 7.60
message: DANGER:large fire! evacuate immediately!((dispatch AIRTANKER))




0: 256x416 2 smokes, 53.2ms
Speed: 14.5ms preprocess, 53.2ms inference, 1.3ms postprocess per image at shape (1, 3, 256, 416)


In [25]:
img_medium = cv2.imread('D://project_deeplearning/annotations/YOLO_UAV/images/test/3.jpg')
h, w = img_medium.shape[:2]
re = model(img_medium, conf=0.5)

boxes = re[0].boxes.xyxy.cpu().numpy()
cls = re[0].boxes.cls.cpu().numpy()

has_fire = any(int(c) == 0 for c in cls)
has_smoke = any(int(c) == 1 for c in cls)

if has_fire and has_smoke:
    print("Result: Both fire and smoke detected")
elif has_fire:
    print("Result: Fire detected only")
elif has_smoke:
    print("Result: Smoke detected only")
else:
    print("Result: No fire or smoke detected")


0: 256x416 2 smokes, 50.6ms
Speed: 9.8ms preprocess, 50.6ms inference, 1.6ms postprocess per image at shape (1, 3, 256, 416)
Result: Smoke detected only


In [26]:
img_medium = cv2.imread('D://project_deeplearning/annotations/YOLO_UAV/images/test/2.jpg')
h, w = img_medium.shape[:2]
re = model(img_medium, conf=0.5)

boxes = re[0].boxes.xyxy.cpu().numpy()
cls = re[0].boxes.cls.cpu().numpy()

has_fire = any(int(c) == 0 for c in cls)
has_smoke = any(int(c) == 1 for c in cls)

if has_fire and has_smoke:
    print("Result: Both fire and smoke detected")
elif has_fire:
    print("Result: Fire detected only")
elif has_smoke:
    print("Result: Smoke detected only")
else:
    print("Result: No fire or smoke detected")


0: 256x416 14 fires, 1 smoke, 50.7ms
Speed: 3.7ms preprocess, 50.7ms inference, 1.5ms postprocess per image at shape (1, 3, 256, 416)
Result: Both fire and smoke detected
